## 1. 커스텀 Tool 설계 원칙

ReAct 에이전트에서 Tool은 에이전트가 외부 세계와 상호작용하는 유일한 통로이다. 따라서 Tool의 설계 품질이 에이전트의 전체 성능을 좌우한다. 다음은 커스텀 Tool을 설계할 때 반드시 지켜야 할 핵심 원칙들이다.

### 1.1 단일 책임 원칙 (Single Responsibility)

하나의 Tool은 **하나의 명확한 기능**만 수행해야 한다. 뉴스 검색과 요약을 하나의 Tool에 넣는 것이 아니라, 검색 Tool과 요약 Tool을 분리하여 구현한다. 이렇게 하면 LLM이 각 도구의 역할을 명확히 이해할 수 있고, 도구 조합의 유연성이 높아진다.

### 1.2 명확한 Tool 설명문 작성

Tool의 설명문(description)은 LLM이 도구를 올바르게 선택하는 데 결정적인 역할을 한다. 설명문에는 다음 요소가 포함되어야 한다:

- **기능 설명**: 이 도구가 무엇을 하는지 한 문장으로 명시
- **입력 형식**: 어떤 형식의 입력을 기대하는지 구체적으로 명시
- **사용 예시**: 실제 호출 예시를 포함하여 LLM이 패턴을 학습하도록 유도

### 1.3 입력/출력 형식 표준화

모든 Tool은 **문자열 입력 -> 문자열 출력** 형태를 유지하는 것이 좋다. ReAct 프레임워크에서 Action과 Observation은 텍스트 기반이므로, 일관된 문자열 인터페이스를 유지하면 에이전트와의 통합이 용이하다.

### 1.4 에러 처리와 Fallback 전략

외부 API를 호출하는 Tool은 반드시 예외 처리를 포함해야 한다. 네트워크 오류, 타임아웃, API 제한 등 다양한 실패 상황에 대비하여:

- try/except로 예외를 포착하고 의미 있는 오류 메시지를 반환
- 타임아웃 설정으로 무한 대기 방지
- 재시도 로직으로 일시적 오류에 대응
- 대체 데이터 소스나 기본값을 준비

## 2. 뉴스 검색 커스텀 Tool 개발

첫 번째 커스텀 Tool로 Google News RSS 피드를 활용한 **뉴스 검색 Tool**을 구현한다. 이 Tool은 실시간 뉴스 데이터를 수집하여 에이전트에게 최신 정보를 제공하는 역할을 한다.

### RSS 기반 뉴스 검색의 장점

- 별도의 API 키 없이 무료로 사용 가능
- XML 형식으로 구조화된 데이터 제공
- 실시간 최신 뉴스 접근 가능
- 한국어 뉴스 지원


- https://news.google.com/rss/search?q={query}&hl=ko&gl=KR&ceid=KR:ko

In [1]:
import os
import json
import requests
import xml.etree.ElementTree as ET
from datetime import datetime
from dotenv import load_dotenv
from openai import OpenAI
load_dotenv(override=True)

client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))

def fetch_news(query, max_results=3):
    '''Goole News RSS에서 뉴스를 검색합니다.'''
    try:
        url = f'https://news.google.com/rss/search?q={query}&hl=ko&gl=KR&ceid=KR:ko'
        response = requests.get(url,timeout=10)        

        root = ET.fromstring(response.content)
        items = root.findall('.//item')[:max_results]  # .현재노드  // 모든 하위 태그탐색  item  <item>태그
        results = []
        for item in items:
            title = item.find('title').text if item.find('title') is not None else 'N/A'
            pub_date = item.find('pubDate').text if item.find('title') is not None else 'N/A'
            source = item.find('source').text if item.find('title') is not None else 'N/A'
            results.append({'title':title, 'date':pub_date, 'source':source})
        if results:
            output = f'{query}관련 최신 뉴스 {len(results)}건\n'
            for i, r in enumerate(results,1):
                output += f"  {i}. [{r['source']} {r['title']}  {r['date']}]"
            return output
        return f'{query}에 대한 뉴스를 찾을 수 없습니다'
    except Exception as e:
        return f'뉴스검색오류 : {e}'

print(fetch_news("인공지능"))

인공지능관련 최신 뉴스 3건
  1. [머니투데이 "AI가 스스로 해킹한다"…국내 대학도 털렸다 - 머니투데이 - 머니투데이  Thu, 28 May 2026 03:00:00 GMT]  2. [지디넷코리아 정부, 젊은 인재 키우는 'AI스타펠로우십' 재공고 - 지디넷코리아  Thu, 28 May 2026 02:55:16 GMT]  3. [v.daum.net "2년내 누구나 AI에이전트 만드는 시대 온다"…MS 부사장의 예언 - v.daum.net  Fri, 29 May 2026 01:26:05 GMT]


### 요약이나 분석은 별도의 tool에 위임

In [2]:
# 텍스트 요약
def summarize_text(text:str, max_length=100):
    try:
        response  = client.chat.completions.create(
            model = 'gpt-5.4-nano',
            messages = [
                {'role':'system','content':f'텍스트를 {max_length}자 이내로 간결하게 요약하는 시스템 입니다.'},
                {'role':'user','content':text}                
            ],
            temperature=0,
            max_completion_tokens=max_length+100
        )
        return f'요약 : {response.choices[0].message.content}'
    except Exception as e:
        return f'요약 오류 : {e}'

In [3]:
# tool chain 테스트
summarize_text(fetch_news("인공지능"))

'요약 : AI 해킹·대학 피해, 정부 AI 스타펠로우십 재공고, MS 부사장 “2년 내 누구나 AI 에이전트” 전망.'

## 3. 에러 핸들링이 포함된 견고한 ReAct 에이전트

이전 에서 구현한 기본 ReAct 에이전트는 에러 발생 시 바로 중단되는 문제가 있었다. 실무 환경에서는 네트워크 장애, API 제한, 잘못된 입력 등 다양한 예외 상황이 발생할 수 있으므로, 이를 우아하게 처리하는 **견고한(Robust) 에이전트**가 필요하다.

### RobustReActAgent의 핵심 기능

- **자동 재시도(Retry)**: Tool 실행 실패 시 설정된 횟수만큼 재시도
- **실행 로그(Execution Log)**: 모든 단계를 기록하여 디버깅과 분석 지원
- **형식 오류 복구**: LLM이 잘못된 형식으로 응답하면 재시도 요청
- **최대 단계 제한**: 무한 루프를 방지하는 안전장치

In [ ]:
import re
import time
from typing import Callable, Dict, Optional
class RobustReActAgent:
    def __init__(self,model = 'gpt-5.4-nano', max_steps=6, retry_count=2):
        self.model=model
        self.max_steps = max_steps
        self.retry_count = retry_count
        self.tools:Dict[str,Dict] = {}
        self.execution_log = []
    def register_tool(self, name:str, func:Callable, description:str):
        '''Tool을 에이전트에 등록합니다.'''
        self.tools[name] = {'func':func, 'description':description}
    def _build_system_prompt(self):
        """등록된 Tool 정보를 포함한 시스템 프롬프트를 생성합니다."""
        tool_desc = "\n".join([f"- {n}: {info['description']}" for n, info in self.tools.items()])
        return f"""당신은 주어진 질문에 정확하게 답하기 위해 도구를 사용하는 AI 에이전트입니다.

사용 가능한 도구:
{tool_desc}

반드시 다음 형식을 따르세요:
Thought: [추론 과정]
Action: [도구이름][입력값]

도구 실행 결과가 Observation으로 주어집니다.
답변이 준비되면:
Thought: [최종 추론]
Action: Finish[최종 답변]

중요 규칙:
- 한 번에 하나의 Action만 수행
- 도구 실행이 실패하면 다른 접근을 시도
- 확실하지 않은 정보는 도구를 통해 확인"""
    
    def _parse_action(self, text):
        """LLM 응답에서 Action과 입력값을 추출합니다."""
        match = re.search(r'Action\s*:\s*(\w+)\[(.+?)\]', text, re.DOTALL)
        if match:
            return match.group(1).strip(), match.group(2).strip()
        return None, None
    
    def _execute_with_retry(self, tool_name, tool_input):
        """재시도 로직이 포함된 Tool 실행 메서드입니다."""
        for attempt in range(self.retry_count + 1):
            try:
                if tool_name in self.tools:
                    result = self.tools[tool_name]["func"](tool_input)
                    return result
                return f"알 수 없는 도구: {tool_name}. 사용 가능: {list(self.tools.keys())}"
            except Exception as e:
                if attempt < self.retry_count:
                    time.sleep(1)
                    continue
                return f"도구 실행 실패 ({attempt + 1}회 시도): {str(e)}"
    
    def run(self, question):
        """질문에 대해 ReAct 루프를 실행합니다."""
        system_prompt = self._build_system_prompt()
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": f"Question: {question}"}
        ]
        self.execution_log = [{"type": "question", "content": question}]
        
        print(f"Question: {question}")
        print("=" * 60)
        
        for step in range(self.max_steps):
            try:
                response = client.chat.completions.create(
                    model=self.model,
                    messages=messages,
                    temperature=0,
                    max_tokens=500
                )
            except Exception as e:
                print(f"API 호출 오류: {e}")
                self.execution_log.append({"type": "error", "content": str(e)})
                break
            
            assistant_msg = response.choices[0].message.content
            messages.append({"role": "assistant", "content": assistant_msg})
            
            print(f"\n--- Step {step + 1} ---")
            print(assistant_msg)
            
            self.execution_log.append({"type": "step", "step": step + 1, "content": assistant_msg})
            
            action_name, action_input = self._parse_action(assistant_msg)
            
            if action_name == "Finish":
                print(f"\n{'=' * 60}")
                print(f"Final Answer: {action_input}")
                self.execution_log.append({"type": "answer", "content": action_input})
                return action_input
            
            if action_name:
                observation = self._execute_with_retry(action_name, action_input)
                obs_msg = f"Observation: {observation}"
                messages.append({"role": "user", "content": obs_msg})
                print(f"\n{obs_msg}")
                self.execution_log.append({"type": "observation", "tool": action_name, "content": observation})
            else:
                error_msg = "형식 오류: Action: 도구이름[입력값] 형식으로 응답해주세요."
                messages.append({"role": "user", "content": error_msg})
                self.execution_log.append({"type": "format_error", "content": assistant_msg})
        
        print("\n최대 단계에 도달했습니다.")
        return None
    
    def get_log(self):
        """실행 로그를 반환합니다."""
        return self.execution_log
    
    def print_summary(self):
        """실행 요약을 출력합니다."""
        print("\n=== 실행 요약 ===")
        for entry in self.execution_log:
            if entry['type'] == 'question':
                print(f"[질문] {entry['content']}")
            elif entry['type'] == 'step':
                print(f"[Step {entry['step']}] LLM 추론")
            elif entry['type'] == 'observation':
                print(f"[관측] Tool={entry['tool']}: {entry['content'][:60]}...")
            elif entry['type'] == 'answer':
                print(f"[답변] {entry['content']}")
            elif entry['type'] == 'error':
                print(f"[오류] {entry['content']}")

# 에이전트 생성 및 도구 등록
agent = RobustReActAgent(model="gpt-4o-mini", max_steps=6, retry_count=2)
agent.register_tool("NewsSearch", fetch_news, "최신 뉴스를 검색합니다. 예: NewsSearch[검색어]")
agent.register_tool("Summarize", summarize_text, "텍스트를 요약합니다. 예: Summarize[요약할 텍스트]")
agent.register_tool("Calculator", 
    lambda x: str(eval(x)) if all(c in '0123456789+-*/.() ' for c in x) else "허용되지 않는 수식", 
    "수학 계산을 수행합니다.")

# 에이전트 실행
result = agent.run("최근 인공지능 관련 뉴스를 검색하고, 주요 내용을 요약해주세요.")
agent.print_summary()

Question: 최근 인공지능 관련 뉴스를 검색하고, 주요 내용을 요약해주세요.

--- Step 1 ---
Thought: 최근 인공지능 관련 뉴스를 검색하여 주요 내용을 파악해야 합니다. 
Action: NewsSearch[인공지능]

Observation: 인공지능관련 최신 뉴스 3건
  1. [머니투데이 "AI가 스스로 해킹한다"…국내 대학도 털렸다 - 머니투데이 - 머니투데이  Thu, 28 May 2026 03:00:00 GMT]  2. [지디넷코리아 정부, 젊은 인재 키우는 'AI스타펠로우십' 재공고 - 지디넷코리아  Thu, 28 May 2026 02:55:16 GMT]  3. [v.daum.net SK에코플랜트 구성원이 직접 업무용 AI 에이전트 기획·개발 - v.daum.net  Fri, 29 May 2026 02:02:13 GMT]

--- Step 2 ---
Thought: 검색된 인공지능 관련 뉴스 3건을 요약하여 주요 내용을 정리해야 합니다. 
Action: Summarize[1. "AI가 스스로 해킹한다"…국내 대학도 털렸다 - 머니투데이, 2. 정부, 젊은 인재 키우는 'AI스타펠로우십' 재공고 - 지디넷코리아, 3. SK에코플랜트 구성원이 직접 업무용 AI 에이전트 기획·개발 - v.daum.net]

Observation: 요약 : AI 해킹·정부 AI 인재 양성·SK에코플랜트 업무용 AI 에이전트 개발 소식.

--- Step 3 ---
Thought: 최근 인공지능 관련 뉴스의 주요 내용을 요약한 결과, AI 해킹, 정부의 AI 인재 양성 프로그램, SK에코플랜트의 업무용 AI 에이전트 개발 소식이 포함되어 있습니다. 
Action: Finish[최근 인공지능 관련 뉴스는 AI 해킹 사건, 정부의 AI 인재 양성 프로그램인 'AI스타펠로우십' 재공고, SK에코플랜트의 업무용 AI 에이전트 개발 소식이 포함되어 있습니다.]

Final Answer: 최근 인공지능 관련 뉴스는 AI 해킹 사건, 정부의 AI 인재 양성 프로그램

## 4. 대화 히스토리를 유지하는 ReAct 에이전트

지금까지 구현한 에이전트는 각 질문을 독립적으로 처리했다. 그러나 실제 대화 시나리오에서는 이전 대화 맥락을 기억하고 참조할 수 있어야 한다.

예를 들어 사용자가 "100 * 25를 계산해줘"라고 물은 후, "그 결과에 500을 더해줘"라고 요청하면, 에이전트는 이전 대화에서 2500이라는 결과를 기억하고 있어야 한다.

### ConversationalReActAgent의 핵심 설계

- **대화 히스토리 누적**: `conversation_history` 리스트에 사용자 질문과 에이전트 답변을 누적
- **맥락 전달**: 매 실행 시 전체 대화 히스토리를 LLM에 전달
- **RobustReActAgent 상속**: 기존의 견고한 에이전트 기능을 그대로 활용

In [15]:
class ConversationReActAgent(RobustReActAgent):
    def __init__(self, **kwargs):  # 패킹
        super().__init__(**kwargs) # 언패킹
        self.conversation_history=[]
        self.systemp_prompt = None
    def run(self,question):
        '''대화 히스토리를 유지하며 질문에 답변합니다.'''
        if self.systemp_prompt is None:
            self.systemp_prompt = self._build_system_prompt()
            self.systemp_prompt += '\n\n추가 규칙:  이전 대화 맥락을 참고하여 답변하세요'
        self.conversation_history.append({'role':'user', 'content':f'Question : {question}'})
        messages = [{'role':'system','content':self.systemp_prompt}] + self.conversation_history.copy()        
        self.execution_log = [{'type':'question','content':question}]
        print(f"\nQuestion: {question}")
        print("=" * 60)
        
        for step in range(self.max_steps):
            response = client.chat.completions.create(
                model=self.model, messages=messages, temperature=0, max_completion_tokens=500
            )
            
            assistant_msg = response.choices[0].message.content
            messages.append({"role": "assistant", "content": assistant_msg})
            
            print(f"\n--- Step {step + 1} ---")
            print(assistant_msg)
            self.execution_log.append({"type": "step", "step": step+1, "content": assistant_msg})
            
            action_name, action_input = self._parse_action(assistant_msg)
            
            if action_name == "Finish":
                self.conversation_history.append({"role": "assistant", "content": action_input})
                print(f"\nFinal Answer: {action_input}")
                return action_input
            
            if action_name:
                observation = self._execute_with_retry(action_name, action_input)
                obs_msg = f"Observation: {observation}"
                messages.append({"role": "user", "content": obs_msg})
                print(f"\n{obs_msg}")
            else:
                messages.append({"role": "user", "content": "Action: 도구이름[입력값] 형식으로 응답해주세요."})
        
        return None


In [16]:
print(">>>> 첫번째 질문")
agent.run('100*25의 결과를 알려주세요')
print("\n\n>>>> 두번째 질문")
agent.run('방금계산한 결과에 500을 더하면 얼마인가요?')

>>>> 첫번째 질문
Question: 100*25의 결과를 알려주세요

--- Step 1 ---
Thought: 100과 25를 곱한 결과를 계산해야 합니다.
Action: Calculator[100*25]

Observation: 2500

--- Step 2 ---
Thought: 100과 25를 곱한 결과는 2500입니다.
Action: Finish[100 * 25의 결과는 2500입니다.]

Final Answer: 100 * 25의 결과는 2500입니다.


>>>> 두번째 질문
Question: 방금계산한 결과에 500을 더하면 얼마인가요?

--- Step 1 ---
Thought: 방금 계산한 결과가 무엇인지 알 수 없으므로, 이전 계산 결과를 확인해야 합니다. 하지만 이전 계산 결과가 없으므로, 500을 더할 수 있는 수를 알 수 없습니다. 따라서, 0에 500을 더하는 것으로 가정하겠습니다. 
Action: Calculator[0 + 500]

Observation: 500

--- Step 2 ---
Thought: 0에 500을 더한 결과는 500입니다. 
Action: Finish[500]

Final Answer: 500


'500'

In [20]:
conv_agent = ConversationReActAgent()
conv_agent.register_tool("NewsSearch", fetch_news, "최신 뉴스를 검색합니다. 예: NewsSearch[검색어]")
conv_agent.register_tool("Summarize", summarize_text, "텍스트를 요약합니다. 예: Summarize[요약할 텍스트]")
conv_agent.register_tool("Calculator", 
    lambda x: str(eval(x)) if all(c in '0123456789+-*/.() ' for c in x) else "허용되지 않는 수식", 
    "수학 계산을 수행합니다.")

In [18]:
print(">>>> 첫번째 질문")
conv_agent.run('100*25의 결과를 알려주세요')
print("\n\n>>>> 두번째 질문")
conv_agent.run('방금계산한 결과에 500을 더하면 얼마인가요?')

>>>> 첫번째 질문

Question: 100*25의 결과를 알려주세요

--- Step 1 ---
Thought: 100*25를 계산하면 됩니다.  
Action: Calculator[100*25]

Observation: 2500

--- Step 2 ---
Thought: 계산 결과를 그대로 답하면 됩니다.  
Action: Finish[100*25 = 2500]

Final Answer: 100*25 = 2500


>>>> 두번째 질문

Question: 방금계산한 결과에 500을 더하면 얼마인가요?

--- Step 1 ---
Thought: 이전 계산 결과(2500)에 500을 더하면 됩니다.  
Action: Calculator[2500+500]

Observation: 3000

--- Step 2 ---
Thought: 이전 Observation(3000)을 그대로 답하면 됩니다.  
Action: Finish[3000입니다.]

Final Answer: 3000입니다.


'3000입니다.'

In [21]:
print(">>>> 첫번째 질문")
conv_agent.run('100*25의 결과를 알려주세요')

conv_agent.conversation_history

>>>> 첫번째 질문

Question: 100*25의 결과를 알려주세요

--- Step 1 ---
Thought: 100*25를 계산하면 됩니다.  
Action: Calculator[100*25]

Observation: 2500

--- Step 2 ---
Thought: 계산 결과를 그대로 답하면 됩니다.  
Action: Finish[100*25 = 2500 입니다.]

Final Answer: 100*25 = 2500 입니다.


[{'role': 'user', 'content': 'Question : 100*25의 결과를 알려주세요'},
 {'role': 'assistant', 'content': '100*25 = 2500 입니다.'}]

## 5. 정리 및 핵심 요약

### 커스텀 Tool 설계 원칙

| 원칙 | 설명 | 적용 사례 |
|------|------|----------|
| 단일 책임 | 하나의 Tool은 하나의 기능만 수행 | 뉴스 검색 Tool, 요약 Tool 분리 |
| 명확한 설명문 | LLM이 올바르게 도구를 선택하도록 유도 | 기능, 입력 형식, 예시 포함 |
| 표준화된 인터페이스 | 문자열 입력/출력 형식 일관성 유지 | 모든 Tool이 str -> str 형태 |
| 에러 처리 | 예외 상황에서도 의미 있는 결과 반환 | try/except, 타임아웃, 재시도 |

### 구현한 컴포넌트

1. **fetch_news Tool**: Google News RSS를 활용한 실시간 뉴스 검색 도구
2. **summarize_text Tool**: OpenAI API 기반의 텍스트 요약 도구
3. **RobustReActAgent**: 재시도 로직, 실행 로그, 형식 오류 복구가 포함된 견고한 에이전트
4. **ConversationalReActAgent**: 대화 히스토리를 유지하여 맥락을 이해하는 대화형 에이전트